# Lesson 1.4 — Multi-Step Workflows: Programmatic Enforcement & Handoff

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Contrast programmatic enforcement (gates/hooks) with prompt-based guidance for ordering
- Implement a prerequisite gate that blocks a tool until a prior step has completed
- Decompose multi-concern requests, investigate in parallel, then synthesize one resolution
- Compile a structured handoff summary for human escalation

**You will produce:** A prerequisite gate (verify-before-refund) plus a structured escalation handoff payload.

## Concepts

When **deterministic compliance** is required - e.g. identity verification before a financial operation - prompt instructions alone have a **non-zero failure rate**. Sample Question 1 describes an agent that skips `get_customer` 12% of the time and refunds the wrong account. The correct fix is a **programmatic prerequisite** that blocks `lookup_order`/`process_refund` until `get_customer` returns a verified customer ID - not a stronger prompt and not few-shot examples.

- **Programmatic enforcement** (hooks, prerequisite gates) gives deterministic ordering.
- **Prompt-based guidance** is probabilistic and insufficient when errors have financial impact.
- **Decompose multi-concern requests** into distinct items, investigate each (in parallel, sharing context), then synthesize one unified resolution.
- **Structured handoff:** when escalating, compile customer ID, root cause, refund amount, and recommended action for a human who cannot see the transcript.

> **Reference:** [Anthropic - Agent SDK hooks & control](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A prerequisite gate (deterministic ordering)

State tracks whether identity is verified. `process_refund` is blocked until it is.

In [ ]:
class Session:
    def __init__(self):
        self.verified_customer_id = None

def get_customer(session, name):
    session.verified_customer_id = "CUST-8842"  # verified lookup
    return session.verified_customer_id

def process_refund(session, amount):
    # GATE: deterministic prerequisite, not a prompt suggestion
    if not session.verified_customer_id:
        raise PermissionError("Blocked: get_customer must verify identity first.")
    return f"Refunded ${amount} to {session.verified_customer_id}"

s = Session()
try:
    process_refund(s, 50)
except PermissionError as e:
    print("Expected block:", e)
get_customer(s, "Dana")
print(process_refund(s, 50))

## Code-Along: Step 2 — Decompose a multi-concern request

A single message can carry several issues. Split them, investigate, then unify.

In [ ]:
def decompose_request(msg):
    # Simple split; in production Claude extracts distinct issues structurally.
    return [p.strip() for p in msg.split(" and ") if p.strip()]

msg = "My order #12345 arrived damaged and I was double-charged on my last invoice"
issues = decompose_request(msg)
print("issues:", issues)
# Each issue investigated with shared, verified customer context, then synthesized.
resolution = {i: "investigated" for i in issues}
print("unified resolution:", resolution)

## Code-Along: Step 3 — Structured escalation handoff

Humans picking up an escalation lack the conversation transcript. Give them a complete, structured summary.

In [ ]:
def build_handoff(session, root_cause, refund_amount, recommended):
    return {
        "customer_id": session.verified_customer_id,
        "root_cause": root_cause,
        "refund_amount": refund_amount,
        "recommended_action": recommended,
    }

handoff = build_handoff(s, "carrier damage; policy exception requested", 50,
                        "approve goodwill refund")
print(json.dumps(handoff, indent=2))

## Exercise

Add an `escalate_to_human(session, handoff)` function that refuses to run unless the handoff dict contains all four required keys (customer_id, root_cause, refund_amount, recommended_action). Then show that a missing `customer_id` (unverified session) raises - enforcing both the identity gate and handoff completeness.

In [ ]:
REQUIRED = {"customer_id", "root_cause", "refund_amount", "recommended_action"}
# TODO: implement escalate_to_human that asserts REQUIRED <= handoff.keys()
# TODO: show it raises on an incomplete handoff

## Key Takeaways

- Deterministic compliance needs programmatic gates; prompts have a non-zero failure rate.
- Block downstream tools (process_refund) until prerequisites (get_customer) complete.
- Decompose multi-concern requests, investigate each with shared context, then unify.
- Escalations need structured handoffs: customer ID, root cause, amount, recommended action.

---

**Next:** [Lesson 1.5 — Agent SDK Hooks: Interception & Normalization](./lesson-1.5-agent-sdk-hooks.ipynb)